# 1. KAFKA & SPARK TUTORIAL
- Môn học: Big Data - CO3137
- Ngày 31/03/2026
- Lớp: L01

| STT | Họ tên | MSSV |
| :---: | :--- | :---: |
| 1 | Lê Đình Đức | 2310774 |
| 2 | Nguyễn Văn Công Thành | 231xxx |

## 1. Khởi tạo Spark Session
Cấu hình Spark để có thể làm việc với Kafka.
Để cấu Spark làm việc với Kafka trong các bài toán xử lý dữ liệu streaming, cần một số denpendencies:
- `spark-sql-kafka` là connector chính cho **Structured Streaming**, cho phép Spark đọc/ghi dữ liệu từ Kafka thông qua API như `readStream`.
- `kafka-clients` là thư viện client cấp thấp dùng để giao tiếp trực tiếp với Kafka broker (được connector sử dụng bên trong).
- `spark-streaming-kafka` phục vụ cho API (connector cho Spark Streaming (DStream API - legacy)) và thường không cần thiết nếu sử dụng Structured Streaming hiện đại.



In [12]:
import kagglehub
from confluent_kafka.admin import AdminClient, NewTopic
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, from_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

spark = (SparkSession.builder.appName("Lab1_Spark_Kafka")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,org.apache.kafka:kafka-clients:3.6.0,org.apache.spark:spark-streaming-kafka-0-10_2.13:4.1.1")
    .config("spark.driver.memory", "4g")
    .master("local[*]")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

## 2. Chuẩn bị Dữ liệu
Tải bộ dữ liệu `movielens-latest-small` từ Kaggle và đọc vào Spark DataFrames.

In [11]:
path = kagglehub.dataset_download("grouplens/movielens-latest-small")

df_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_tags = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)

print("Ratings preview:")
df_ratings.show(3)
print("Movies preview:")
df_movies.show(3)
print("Tags preview:")
df_tags.show(3)


Ratings preview:
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
+------+-------+------+---------+
only showing top 3 rows
Movies preview:
+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
+-------+--------------------+--------------------+
only showing top 3 rows
Tags preview:
+------+-------+---------------+----------+
|userId|movieId|            tag| timestamp|
+------+-------+---------------+----------+
|     2|  60756|          funny|1445714994|
|     2|  60756|Highly quotable|1445714996|
|     2|  60756|   will ferrell|1445714992|
+------+-------+---------------+-------

## 3. Thiết lập Kafka Topics
Định nghĩa 3 brokers và tạo các topics tương ứng (`ratings`, `movies`, `tags`).

In [13]:
KAFKA_BROKERS = "localhost:9092,localhost:9192,localhost:9292"

# Delete existing topics and create them anew
admin_client = AdminClient({'bootstrap.servers': KAFKA_BROKERS})
admin_client.delete_topics(['ratings', 'movies', 'tags'], operation_timeout=10)
print("Deleted existing topics (if any)")

new_topics = [
    NewTopic(topic="ratings", num_partitions=3, replication_factor=2),
    NewTopic(topic="movies", num_partitions=3, replication_factor=2),
    NewTopic(topic="tags", num_partitions=3, replication_factor=2)
]
fs = admin_client.create_topics(new_topics)
for topic, f in fs.items():
    try:
        f.result() # Wait for topic generation
        print(f"Topic '{topic}' created successfully.")
    except Exception as e:
        print(f"Failed to create topic '{topic}' (Might ignore if it's due to existing topic): {e}")


Deleted existing topics (if any)
Topic 'ratings' created successfully.
Topic 'movies' created successfully.
Topic 'tags' created successfully.


## 4. Đưa Dữ Liệu Lên Kafka
Đẩy dữ liệu từ Spark DataFrames vào Kafka. DataFrames cần được parse dưới dạng JSON vào cột `value`.

In [14]:
df_ratings.selectExpr("to_json(struct(*)) AS value") \
    .write.format("kafka").option("kafka.bootstrap.servers", KAFKA_BROKERS) \
    .option("topic", "ratings").save()

df_movies.selectExpr("to_json(struct(*)) AS value") \
    .write.format("kafka").option("kafka.bootstrap.servers", KAFKA_BROKERS) \
    .option("topic", "movies").save()

df_tags.selectExpr("to_json(struct(*)) AS value") \
    .write.format("kafka").option("kafka.bootstrap.servers", KAFKA_BROKERS) \
    .option("topic", "tags").save()

print("Successfully written data to Kafka topics.")


Successfully written data to Kafka topics.


## 5. Đọc Dữ Liệu Từ Kafka & Xử Lý Lỗi Định Dạng

Dữ liệu khi đọc từ **Apache Kafka** trong **Apache Spark** thường có dạng nhị phân (`binary`) ở cột `value`, nên không thể xử lý trực tiếp như dữ liệu có cấu trúc. Để khắc phục, ta cần ép kiểu (`cast`) cột `value` sang `STRING`, sau đó sử dụng hàm `from_json` kết hợp với schema đã định nghĩa trước để parse chuỗi JSON này thành các cột có kiểu dữ liệu rõ ràng. Cách này giúp Spark hiểu đúng cấu trúc dữ liệu và tránh các lỗi liên quan đến định dạng khi xử lý streaming.

**`StructType`** trong Apache Spark là một kiểu dữ liệu thuộc hệ thống schema (cụ thể là `pyspark.sql.types`), dùng để định nghĩa **cấu trúc của một DataFrame** dưới dạng tập hợp các cột có kiểu dữ liệu xác định. Nó bao gồm nhiều `StructField`, trong đó mỗi field mô tả tên cột, kiểu dữ liệu (như `StringType`, `IntegerType`, …) và khả năng null (`nullable`). Trong các bài toán như đọc dữ liệu từ Kafka (JSON dạng string), `StructType` đóng vai trò rất quan trọng khi kết hợp với hàm `from_json` để **parse dữ liệu bán cấu trúc thành dạng có schema rõ ràng**, giúp Spark tối ưu hóa xử lý và tránh lỗi định dạng.


In [16]:
# Define Schemas
movie_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

rating_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", IntegerType(), True)
])

tag_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", IntegerType(), True)
])

def read_and_parse_kafka(topic, schema):
    """Read from Kafka and parse JSON string"""
    df_raw = spark.read \
        .format("kafka") \
        .option("kafka.bootstrap.servers", KAFKA_BROKERS) \
        .option("subscribe", topic) \
        .option("startingOffsets", "earliest") \
        .load()
    
    # Resolve data format: CAST(value AS STRING)
    df_parsed = df_raw.selectExpr("CAST(value AS STRING) as json_str") \
        .select(from_json(col("json_str"), schema).alias("data")) \
        .select("data.*")
    
    return df_parsed

df_movies_parsed = read_and_parse_kafka("movies", movie_schema)
df_ratings_parsed = read_and_parse_kafka("ratings", rating_schema)
df_tags_parsed = read_and_parse_kafka("tags", tag_schema)

print("Parsed Movie Data:")
df_movies_parsed.show(5)

print("Parsed Ratings Data:")
df_ratings_parsed.show(5)

print("Parsed Tags Data:")
df_tags_parsed.show(5)


Parsed Movie Data:
+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|   1022|   Cinderella (1950)|Animation|Childre...|
|   1023|Winnie the Pooh a...|Animation|Childre...|
|   1024|Three Caballeros,...|Animation|Childre...|
|   1025|Sword in the Ston...|Animation|Childre...|
|   1027|Robin Hood: Princ...|     Adventure|Drama|
+-------+--------------------+--------------------+
only showing top 5 rows
Parsed Ratings Data:
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows
Parsed Tags Data:
+------+-------+-----------+----------+
|userId|movieId|        tag| timestamp|
+------+-------+-----------+----------+


## 6. Lấy Top 5 Phim Có Xếp Hạng Cao Nhất 
Lấy danh sách 5 bộ phim có điểm rating trung bình cao nhất, với điều kiện có số lượng người đánh giá > 30.

In [23]:
# Get Top 5 movies in df_ratings
top_5_movies = df_ratings_parsed.groupBy("movieId") \
    .agg(
        avg("rating").alias("avg_rating"),
        count("rating").alias("rating_count")
    ) \
    .filter(col("rating_count") > 30) \
    .limit(5)

# Join with movies to get tiltle
top_5_movies_with_name = top_5_movies.join(df_movies_parsed, on="movieId", how="inner") \
    .select("movieId", "title", "avg_rating", "rating_count") \
    .orderBy(col("avg_rating").desc(), col("movieId").asc())

print("Top 5 movies (with titles) whose rating count > 30:")
top_5_movies_with_name.show(truncate=False)

Top 5 movies (with titles) whose rating count > 30:


+-------+--------------------------------+-----------------+------------+
|movieId|title                           |avg_rating       |rating_count|
+-------+--------------------------------+-----------------+------------+
|3175   |Galaxy Quest (1999)             |3.58             |75          |
|471    |Hudsucker Proxy, The (1994)     |3.55             |40          |
|1580   |Men in Black (a.k.a. MIB) (1997)|3.487878787878788|165         |
|1645   |The Devil's Advocate (1997)     |3.411764705882353|51          |
|1088   |Dirty Dancing (1987)            |3.369047619047619|42          |
+-------+--------------------------------+-----------------+------------+



## 7. Tìm 5 Tag Tệ Nhất
Tìm 5 tags liên kết với điểm rating trung bình thấp nhất. Ta sẽ thực hiện kết nối (join) bảng tag và rating qua `movieId`.

In [18]:
df_tags_ratings = df_tags_parsed.join(df_ratings_parsed, on="movieId", how="inner")

worst_5_tags = df_tags_ratings.groupBy("tag") \
    .agg(avg("rating").alias("tag_avg_rating")) \
    .orderBy(col("tag_avg_rating").asc()) \
    .limit(5)

print("5 Worst Tags (lowest average rating):")
worst_5_tags.show()

# Extract tag names to list for the next query
worst_tags_list = [row['tag'] for row in worst_5_tags.collect()]
print(f"List of worst tags: {worst_tags_list}")


5 Worst Tags (lowest average rating):


+--------+------------------+
|     tag|    tag_avg_rating|
+--------+------------------+
|symbolic|               0.5|
|   shark|1.4166666666666667|
|   stage|              1.75|
|   Tokyo|               2.0|
|     SNL|               2.1|
+--------+------------------+



List of worst tags: ['symbolic', 'shark', 'stage', 'Tokyo', 'SNL']


## 8. Phân Tích Sự Tác Động Của Tag
Câu hỏi: "Các bộ phim mang các tag trên có xu hướng nhận điểm thấp hay không? Với mỗi phim có tag đó, kiểm tra điểm đánh giá trung bình và số lượng phim có mỗi tag."

In [20]:
# Filter records of movies that contain the worst tags
df_worst_tags_movies = df_tags_parsed.filter(col("tag").isin(worst_tags_list))


# Check average ratings for EACH movie associated with these tags
movie_tag_ratings = df_worst_tags_movies.select("movieId", "tag").distinct() \
    .join(df_ratings_parsed, on="movieId", how="inner") \
    .groupBy("tag", "movieId") \
    .agg(avg("rating").alias("movie_avg_rating")) \
    .orderBy("tag", col("movie_avg_rating").asc())

print("Average ratings for each movie associated with the worst tags:")
movie_tag_ratings.show(15)

# Check how many movies have these tags
movies_per_worst_tag = df_worst_tags_movies.select("movieId", "tag").distinct() \
    .groupBy("tag") \
    .agg(count("movieId").alias("num_movies_with_tag")) \
    .orderBy(col("num_movies_with_tag").desc())

print("How many movies have these tags:")
movies_per_worst_tag.show()


# 4. Overall analysis (Use .first()[0] to directly extract the scalar value, avoiding .collect() list)
overall_avg = df_ratings_parsed.agg(avg("rating")).first()[0] or 0

worst_movies_ratings = df_worst_tags_movies.select("movieId").distinct() \
    .join(df_ratings_parsed, on="movieId", how="inner")

worst_tags_overall_avg = worst_movies_ratings.agg(avg("rating")).first()[0] or 0

print(f"Overall Average Rating in complete Dataset: {overall_avg:.2f}")
print(f"Average Rating of movies with the worst tags: {worst_tags_overall_avg:.2f}")

if worst_tags_overall_avg < overall_avg:
    print("=> Conclusion: Yes, indeed! Movies with these tags tend to receive lower ratings than average.")
else:
    print("=> Conclusion: Not necessarily, deeper analysis is required.")

Average ratings for each movie associated with the worst tags:


+--------+-------+------------------+
|     tag|movieId|  movie_avg_rating|
+--------+-------+------------------+
|     SNL|   2296|               2.1|
|   Tokyo|   6407|               2.0|
|   shark|   1389|1.4166666666666667|
|   stage|   8943|              1.75|
|symbolic|  26717|               0.5|
+--------+-------+------------------+

How many movies have these tags:


+--------+-------------------+
|     tag|num_movies_with_tag|
+--------+-------------------+
|   Tokyo|                  1|
|   shark|                  1|
|     SNL|                  1|
|symbolic|                  1|
|   stage|                  1|
+--------+-------------------+



Overall Average Rating in complete Dataset: 3.50
Average Rating of movies with the worst tags: 1.77
=> Conclusion: Yes, indeed! Movies with these tags tend to receive lower ratings than average.
